In [ ]:
import pandas as pd
import itertools
import random
from collections import defaultdict
from sklearn.model_selection import train_test_split

In [ ]:
INPUT_DIR = './data'
df = pd.read_json(f'{INPUT_DIR}/final_dataset_gt.jsonl', lines=True)
sub_df = pd.read_json(f'{INPUT_DIR}/submissions.jsonl', lines=True)
df['id'] = df['sub_id'].str.rsplit('_', n=1).str[0]

df = df.merge(
    sub_df[['sub_id', 'code', 'lang']],
    left_on='sub_id',
    right_on='sub_id',
    how='left'
)

In [ ]:
pairwise_records = []
criteria_cols = ['efficiency', 'correctness', 'readability']

for problem_id, group in df.groupby('id'):
    submissions = group.to_dict('records')
    
    # Tạo tất cả các cặp
    for sub1, sub2 in itertools.combinations(submissions, 2):
        
        # Duyệt qua từng tiêu chí
        for criteria in criteria_cols:
            score_col = f'{criteria}_score'
            score1 = sub1[score_col]
            score2 = sub2[score_col]
            
            # Logic gán nhãn mới (bao gồm trường hợp bằng nhau)
            if score1 > score2:
                label = 1  # code1 tốt hơn
            elif score1 < score2:
                label = 0  # code2 tốt hơn
            else:
                label = 0.5  # hòa (điểm bằng nhau)
                
            # Đưa thẳng vào list mà không cần điều kiện lọc khác điểm nữa
            pairwise_records.append({
                'id': problem_id,
                'criteria': criteria,
                'sub_id_1': sub1['sub_id'],
                'sub_id_2': sub2['sub_id'],
                'label': label
            })

pairwise_df = pd.DataFrame(pairwise_records)
pairwise_df

In [ ]:
unique_problems = pairwise_df['id'].unique()
train_problems, test_problems = train_test_split(unique_problems, test_size=0.2, random_state=42)

train_df = pairwise_df[pairwise_df['id'].isin(train_problems)].copy()
test_df = pairwise_df[pairwise_df['id'].isin(test_problems)].copy()

train_df.to_json(f'{INPUT_DIR}/pairwise_train.jsonl', orient='records', lines=True)
test_df.to_json(f'{INPUT_DIR}/pairwise_test.jsonl', orient='records', lines=True)